### Imports and hardware checks

In [ ]:
using Pkg
Pkg.activate("../")
Base.Threads.nthreads()

In [ ]:
using Random, Plots, StatsBase, JLD2, OMEinsum, SparseArrays, LinearAlgebra
using OhMyU1
using OhMyU1: ChargeIndex, U1MPS
using OhMyU1: sample_feasible_lp, sample_feasible_lp!, compute_link_charges, compute_site_charges, compute_indices, init_u1_mps, 
orthogonalize!, normalize!, TrainParams, boltzman_probability, train_nondeg!, sample_nondeg_parallel, cost, u1_norm, get_mps
using OhMyU1: diversify!, update_index!, graph_dist, random_bilinear_form, random_assignment_vector, fill_assignment_vectors_parallel!,
pick_best_samples, assignment_matrix, generate_random_matrix_vector, SolverParams, graph_dist_global, solve, get_mps, build_mps_from_feasible_samples,
OptimizationProblem, assert_feasible_lp, SolverStatistics, fill_assignment_vectors!, sort_and_truncate, train_step!, sample_nondeg_parallel!
using Statistics, LaTeXStrings, HypothesisTests
using Base.Threads: @spawn
using JuMP, SCIP

### Func tools

In [21]:
function solve_simple(opt_problem::OptimizationProblem, mps::U1MPS, T::Matrix{Int}, solver_params::SolverParams; parallel::Bool=true, print_stats::Bool=true)

    sorting_crit = "best_cost"

    sp = solver_params
    cost_function = opt_problem.cost_function
    n_vars, _ = opt_problem.num_vars, opt_problem.num_constraints

    # Metrics
    solver_stats = SolverStatistics()

    # Initial feasible dataset
    T, costs = sort_and_truncate(T, cost_function, sp.NUM_FEASIBLE_SAMPLES)

    num_to_keep = floor(Int, sp.UTILITY_FRACTION * size(T, 2))
    best_samples = T[:, 1:num_to_keep]
    best_costs = mapslices(cost_function, best_samples, dims=1)
    solver_stats.c_min = minimum(best_costs)
    solver_stats.incub = best_samples[:, 1]

    push!(solver_stats.learning_curve, solver_stats.c_min)
    push!(solver_stats.utility_curve, mean(best_costs))

    # Memory BUF for samples from MPS to process:
    samples = Matrix{Int}(undef, n_vars, sp.NUM_MPS_SAMPLES + size(best_samples, 2))  #TODO: memory allocation;

    if print_stats
        println("Initial c_min: ", solver_stats.c_min)
        println("Initial utility: ", solver_stats.utility_curve[end])
        println("Strategy for picking new samples: ", sorting_crit)
        println("Diversification: ", diversify)
        println("Barrier: ", barrier)
    end

    # Main loop
    for k in 1:sp.NUM_GLOBAL_ITER

        if print_stats
            println("\n=== Global Iteration ", k, " ===")
        end

        # MPS initialization (with diversification if specified)
        orthogonalize!(mps)
        solver_stats.num_samples_in_mps = Int(round(u1_norm(mps) ^ 2))
        normalize!(mps)

        # Learning step
        temperature = std(costs)
        train_step!(mps, T, sp, temperature, cost_function, solver_stats.learning_curve[end])
        solver_stats.temperature = temperature

        # Sampling
        fill!(samples, 0)  # clear memory buf;
        if parallel
            sample_nondeg_parallel!(mps, samples, sp.NUM_MPS_SAMPLES)
        else
            sample_nondeg!(mps, samples, sp.NUM_MPS_SAMPLES)
        end

        # Keep best samples from previous iteration
        for (j, best_x) in enumerate(eachcol(best_samples))
            samples[:, sp.NUM_MPS_SAMPLES + j] .= best_x
        end

        # Unique samples
        T = unique(eachcol(samples)) |> x -> reduce(hcat, x)
        solver_stats.num_unique_samples = size(T, 2)

        # Sort and keep best feasible, then - add worst samples for diversification;
        T, costs = sort_and_truncate(T, cost_function, size(T, 2))
        best_samples = T[:, 1:num_to_keep]
        best_costs = mapslices(cost_function, best_samples, dims=1)
        solver_stats.c_min = minimum(best_costs)
        solver_stats.incub = best_samples[:, 1]

        # Update learning curve and statistics:
        push!(solver_stats.learning_curve, solver_stats.c_min)
        push!(solver_stats.utility_curve, mean(best_costs))
        
        best_set = Set(eachcol(best_samples))
        keep_indices = filter(i -> view(T, :, i) ∉ best_set, axes(T, 2))
        T = hcat(best_samples, view(T, :, keep_indices))  # Not to loose best (in cost) samples;
        T, costs = sort_and_truncate(T, cost_function, size(T, 2))
        best_samples = T[:, 1:num_to_keep]

        if print_stats
            print_statistics(solver_stats)
        end

    end

    return solver_stats
end

solve_simple (generic function with 1 method)

### Time to solve the assignment (orthogonalize + normalize + sample)

In [41]:
t_arr = []
n_arr = [4, 4, 5, 6, 7, 8, 9, 10]
for n in n_arr
    A = assignment_matrix(n)
    b = ones(Int, 2 * n)
    Q = random_bilinear_form(n^2, 1.0, 7)
    X = Matrix{Int}(undef, n^2, 100000)
    fill_assignment_vectors!(X, n)
    opt_problem = OptimizationProblem(A=A, b=b, cost_function=x->x'*Q*x)
    mps = build_mps_from_feasible_samples(opt_problem, X, 1)
    T = X[:, 1:400]
    t = @elapsed begin
        solve_simple(opt_problem, mps, T, SolverParams(NUM_FEASIBLE_SAMPLES=400, NUM_MPS_SAMPLES=400, NUM_GLOBAL_ITER=1), parallel=true, print_stats=false)
    end
    push!(t_arr, t)
end

In [42]:
t_arr

8-element Vector{Any}:
 0.6245076
 0.0158833
 0.030722
 0.0561826
 0.119244
 0.1651943
 0.3518457
 0.88842

In [19]:
r = maximum([length(mps.LinkIndices[i].Charges) for i in 1:length(mps.LinkIndices)])

461